# Session 1: a rule that splits two groups
**A. Real-world problem.** A packing line must send ripe fruit to the shop and keep unripe
fruit back. A camera gives two numbers per fruit: its size (cm) and a colour score
(0 = green, 10 = deep red). A *classifier* is a rule that turns those two numbers into a
decision. Today you write rules, draw their boundary, and count how they go wrong.
(General education only, with made-up data.)

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

# Twelve fruit by hand: (size cm, colour score, label). label 1 = ripe, 0 = unripe.
TOY = [
    (2.0, 2.0, 0), (3.0, 4.0, 0), (4.0, 2.0, 0), (3.5, 6.0, 0), (5.0, 3.0, 0), (6.0, 4.5, 0),
    (4.5, 7.0, 1), (6.0, 8.0, 1), (7.0, 6.0, 1), (8.0, 9.0, 1), (7.5, 4.0, 1), (5.5, 5.5, 1),
]

## Step 1: a rule is a yes/no question
The simplest rule asks one question about one number: "is the size above t?"
A better rule asks about a mix of both numbers. The set of points where the answer flips
is the **decision boundary**. For the line rule below, the boundary is the straight line

    x * cos(a) + y * sin(a) = c

where x is size, y is colour, a is the angle of the line's normal, and c is how far the
line sits from the origin. Fruit on the far side are called ripe (1), others unripe (0).

In [ ]:
def size_rule(points, t):
    """Predict 1 when size is above t (one number only)."""
    return (np.asarray(points)[:, 0] > t).astype(int)


def line_rule(points, angle_deg, c):
    """Predict 1 when x*cos(a) + y*sin(a) > c."""
    a = np.radians(angle_deg)
    p = np.asarray(points)
    return (p[:, 0] * np.cos(a) + p[:, 1] * np.sin(a) > c).astype(int)

## Step 2: two kinds of mistake
Call "ripe" the thing we are looking for ("positive").
* **False alarm** (false positive): we said ripe, the fruit is unripe.
* **Miss** (false negative): we said unripe, the fruit is ripe.
They cost different things, so we count them separately.

In [ ]:
def count_mistakes(y_true, y_pred):
    """Return (false_alarms, misses)."""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    false_alarms = int(np.sum((y_pred == 1) & (y_true == 0)))
    misses = int(np.sum((y_pred == 0) & (y_true == 1)))
    return false_alarms, misses


def accuracy(y_true, y_pred):
    return float(np.mean(np.asarray(y_true) == np.asarray(y_pred)))


def toy_arrays():
    arr = np.array(TOY, dtype=float)
    return arr[:, :2], arr[:, 2].astype(int)

## Step 3: a bigger, random batch
Real fruit vary, so the two groups overlap. `make_fruit` draws two clusters from normal
distributions with a fixed seed, so everyone gets the same picture.

In [ ]:
def make_fruit(n_each=100, seed=3):
    rng = np.random.default_rng(seed)
    unripe = rng.normal([3.0, 3.0], 1.3, size=(n_each, 2))
    ripe = rng.normal([6.5, 6.5], 1.3, size=(n_each, 2))
    X = np.vstack([unripe, ripe])
    y = np.r_[np.zeros(n_each, int), np.ones(n_each, int)]
    return X, y


def sweep_offset(X, y, angle_deg, cs):
    """False alarms and misses as the line slides: the trade-off curve."""
    fa, ms = [], []
    for c in cs:
        f, m = count_mistakes(y, line_rule(X, angle_deg, c))
        fa.append(f)
        ms.append(m)
    return np.array(fa), np.array(ms)

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    Xt, yt = toy_arrays()
    pred = line_rule(Xt, 45, 7.0)
    print("toy: mistakes (false alarms, misses) =", count_mistakes(yt, pred), " accuracy", round(accuracy(yt, pred), 3))

    X, y = make_fruit()
    for name, pr in [("size only (t = 4.75)", size_rule(X, 4.75)), ("size + colour line", line_rule(X, 45, 6.7))]:
        fa, ms = count_mistakes(y, pr)
        print(f"{name:24s} false alarms {fa:3d}  misses {ms:3d}  accuracy {accuracy(y, pr):.3f}")

    fig, ax = plt.subplots(figsize=(5.5, 5))
    pr = line_rule(X, 45, 6.7)
    ax.scatter(*X[y == 0].T, c="tab:green", s=14, label="unripe")
    ax.scatter(*X[y == 1].T, c="tab:red", s=14, label="ripe")
    wrong = pr != y
    ax.scatter(*X[wrong].T, facecolors="none", edgecolors="k", s=60, label="mistake")
    xs = np.linspace(0, 10, 2)
    ax.plot(xs, (6.7 - xs * np.cos(np.radians(45))) / np.sin(np.radians(45)), "k--", label="boundary")
    ax.set_xlim(0, 10); ax.set_ylim(0, 10)
    ax.set_xlabel("size (cm)"); ax.set_ylabel("colour score (0 green to 10 red)")
    ax.set_title("Two clusters, one boundary, some mistakes")
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
    fig.tight_layout(); fig.savefig(OUT / "clusters_boundary.png", dpi=120)

    cs = np.linspace(2, 11, 91)
    fa, ms = sweep_offset(X, y, 45, cs)
    fig2, ax2 = plt.subplots(figsize=(6, 4))
    ax2.plot(cs, fa, label="false alarms")
    ax2.plot(cs, ms, label="misses")
    ax2.set_xlabel("line offset c (score units)"); ax2.set_ylabel("count (out of 100 each)")
    ax2.set_title("Sliding the boundary trades one mistake for the other")
    ax2.legend(); ax2.grid(alpha=0.3)
    fig2.tight_layout(); fig2.savefig(OUT / "two_mistakes.png", dpi=120)
    print("c with equal counts near:", cs[np.argmin(np.abs(fa - ms))])

## Coding tasks
1. **Easy.** In `lesson.py` change the angle of the line rule to 30 degrees and find, by trying a few offsets, a c that gives no more than 8 mistakes on `make_fruit()`.
2. **Stretch.** Write `best_offset(X, y, angle)` that returns the offset c with the fewest total mistakes, and print it for 45 degrees.
3. **Explain.** Explain why a fruit exactly on the boundary is a problem for any rule, and how the code decides it.

## Assessment
- **A. Conceptual.** Why do we count false alarms and misses separately instead of just counting 'mistakes'? Give an example where one is worse than the other.
- **B. Numerical.** For the twelve fruit of the worked example, use the size-only rule 'ripe if size > 5' (that is a = 0 degrees, c = 5). Count the false alarms, the misses and the accuracy.
- **C. Graph and simulation interpretation.** In the trade-off graph for 200 random fruit with the 45 degree boundary, what happens to the two counts as the offset c increases from 2 to 11? Where do they cross?
- **D. Design and optimisation challenge.** Design: the shop says a false alarm (unripe fruit on the shelf) is worse than a miss. Should you move the boundary left or right of c = 6.7? What do you give up? How would you decide how far?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    X, y = L.toy_arrays()
    checks.append(("toy data: 12 fruit, 6 ripe", len(y) == 12 and int(y.sum()) == 6))

    # hand calculation: x*cos45 + y*sin45 > 7 means x + y > 9.899; only (6, 4.5) with sum 10.5 is wrongly ripe
    fa, ms = L.count_mistakes(y, L.line_rule(X, 45, 7.0))
    checks.append((f"worked example: line (45 degrees, c = 7) gives 1 false alarm and 0 misses (got {fa}, {ms})", (fa, ms) == (1, 0)))
    checks.append(("worked example: accuracy 11/12 = 0.917", abs(L.accuracy(y, L.line_rule(X, 45, 7.0)) - 11 / 12) < 1e-9))
    checks.append(("a perfect line exists for the toy fruit (45 degrees, c = 7.6)", L.count_mistakes(y, L.line_rule(X, 45, 7.6)) == (0, 0)))
    checks.append(("size-only rule at 5.25 makes 1 false alarm and 1 miss", L.count_mistakes(y, L.size_rule(X, 5.25)) == (1, 1)))

    checks.append(("size-only rule at 5 (size > 5): 1 false alarm, 1 miss, accuracy 10/12", L.count_mistakes(y, L.size_rule(X, 5.0)) == (1, 1) and abs(L.accuracy(y, L.size_rule(X, 5.0)) - 10 / 12) < 1e-9))
    # edge cases: a fruit exactly on the boundary is called unripe (strict inequality)
    checks.append(("a point exactly on the boundary is called 0", int(L.size_rule(np.array([[5.0, 1.0]]), 5.0)[0]) == 0))
    checks.append(("a boundary far right calls nothing ripe", int(L.line_rule(X, 0, 100).sum()) == 0))
    checks.append(("a boundary far left calls everything ripe", int(L.line_rule(X, 0, -100).sum()) == 12))

    Xb, yb = L.make_fruit()
    checks.append(("make_fruit is reproducible with its seed", np.allclose(Xb, L.make_fruit()[0])))
    fa1, ms1 = L.count_mistakes(yb, L.size_rule(Xb, 4.75))
    fa2, ms2 = L.count_mistakes(yb, L.line_rule(Xb, 45, 6.7))
    checks.append((f"size only: 14 false alarms and 8 misses of 200 (got {fa1}, {ms1})", (fa1, ms1) == (14, 8)))
    checks.append((f"size + colour line: 3 false alarms and 4 misses (got {fa2}, {ms2})", (fa2, ms2) == (3, 4)))
    checks.append(("using colour as well lowers the total mistakes", fa2 + ms2 < fa1 + ms1))
    checks.append(("accuracy of the line rule is 0.965", abs(L.accuracy(yb, L.line_rule(Xb, 45, 6.7)) - 0.965) < 1e-9))

    cs = np.linspace(2, 11, 91)
    fa_c, ms_c = L.sweep_offset(Xb, yb, 45, cs)
    checks.append(("sliding the line right: false alarms never rise, misses never fall", bool(np.all(np.diff(fa_c) <= 0) and np.all(np.diff(ms_c) >= 0))))
    checks.append((f"at c = 2: 96 false alarms and 0 misses (got {fa_c[0]}, {ms_c[0]})", (int(fa_c[0]), int(ms_c[0])) == (96, 0)))
    checks.append(("the two kinds of mistake are equal near c = 6.6", abs(cs[int(np.argmin(np.abs(fa_c - ms_c)))] - 6.6) < 0.2))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
